# Phase 2 — Fine-Tuning & Evaluation (Module 2)

**Project:** Automated Code Review AI with Vulnerability & Style Feedback
**Prereq:** run `01_data_prep_and_baseline.ipynb` first — it writes the prepared
Python/JS splits to Drive (`code-review-ai/data/prepared/{train,valid,test}.pyjs.jsonl`).

**This notebook (run top-to-bottom on a GPU runtime):**
1. Load the prepared splits.
2. Fine-tune **CodeT5+ (`Salesforce/codet5p-220m`)** on comment generation (diff → review comment).
3. Evaluate **BLEU-4** (and **CodeBLEU**, with the caveat noted below) on the test split.
4. Compare vs. the paper baseline (**14.8** BLEU-4) and this project's targets (**≥18 / ≥22**).
5. Export fine-tuned weights + "after" example outputs (pairs with Phase 1's baseline for the report).

> **Proof-of-work:** run every cell, then commit this notebook **with outputs intact**.
> **Colab Pro** recommended — removes the session-timeout risk on the training run.

> **Metric caveat (flagged):** BLEU-4 is the correct metric for *comment generation*
> (it's what the base paper reports for the `msg` task). **CodeBLEU** is defined for
> *code generation* (it uses AST + data-flow) and is what the paper uses for the
> *code-refinement* (`ref`) task — applying it to natural-language review comments is
> unusual. We compute it for completeness against the stated target, but BLEU-4 is the
> metric to trust for this task. See the note before the CodeBLEU cell.


## 0. Environment & config


In [ ]:
# Pin transformers to 4.44.2 (newer breaks the CodeT5+ tokenizer). If this cell
# DOWNGRADES transformers, do Runtime -> Restart session, then Run all again.
# tree-sitter-* are the grammars CodeBLEU needs; nltk provides smoothed BLEU.
!pip -q install "transformers==4.44.2" "datasets>=2.20" "accelerate>=0.33" \
                "evaluate>=0.4" "sacrebleu>=2.4" "nltk>=3.8" \
                "codebleu>=0.7" "tree-sitter-python" "tree-sitter-javascript"
import torch, transformers, platform
print("torch:", torch.__version__, "| transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"))


In [ ]:
import os, json, glob, random, textwrap
from pathlib import Path

MODEL_NAME = "Salesforce/codet5p-220m"
MAX_SRC, MAX_TGT = 512, 128
SEED = 42
random.seed(SEED); torch.manual_seed(SEED)

# --- Training hyperparameters (prototype defaults; tune for final metrics) -----
EPOCHS      = 3
LR          = 5e-5
TRAIN_BS    = 8
EVAL_BS     = 8
GRAD_ACCUM  = 2
# For a fast Aug-2 prototype, cap the training set. Set MAX_TRAIN = None to use ALL
# prepared Python/JS records (better metrics, longer run — use Colab Pro).
MAX_TRAIN   = 20000
GEN_BEAMS   = 4

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/code-review-ai")
else:
    ROOT = Path("/content/code-review-ai")

PREP_DIR = ROOT / "data" / "prepared"
ART_DIR  = ROOT / "artifacts"
CKPT_DIR = ROOT / "checkpoints" / "codet5p-220m-msg-pyjs"
WEIGHTS_DIR = ROOT / "weights" / "codet5p-220m-msg-pyjs"
for d in (ART_DIR, CKPT_DIR, WEIGHTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
assert PREP_DIR.exists(), f"Missing {PREP_DIR} — run notebook 01 first."
print("ROOT =", ROOT)


## 1. Load prepared splits (diff → comment)


In [ ]:
def read_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

# train = multilingual pool (train.all.jsonl); valid/test = Python/JS (*.pyjs.jsonl).
# Prefer .all for train, .pyjs for eval; skip empty files (e.g. a stale 0-row file).
def load_split(name):
    for fn in (f"{name}.all.jsonl", f"{name}.pyjs.jsonl"):
        p = PREP_DIR / fn
        if p.exists():
            rows = read_jsonl(p)
            if rows:
                return rows, fn
    return [], None

splits = {}
for s in ("train", "valid", "test"):
    splits[s], src = load_split(s)
    print(f"{s}: {len(splits[s])} records  <- {src}")
assert splits["train"], "No training data — run notebook 01 (writes train.all.jsonl)."

# Detect the input (diff) and target (comment) field names from a sample.
sample = splits["train"][0] if splits["train"] else splits["test"][0]
DIFF_KEY = next((k for k in sample if k.lower() in
                 ("diff_hunk", "patch", "diff", "old_file")), "diff_hunk")
GOLD_KEY = next((k for k in sample if k.lower() in ("comment", "msg", "target")), "comment")
print("input field:", DIFF_KEY, "| target field:", GOLD_KEY)

if MAX_TRAIN is not None and len(splits["train"]) > MAX_TRAIN:
    random.shuffle(splits["train"])
    splits["train"] = splits["train"][:MAX_TRAIN]
    print(f"Subsampled train -> {len(splits['train'])} (MAX_TRAIN={MAX_TRAIN})")


In [ ]:
from datasets import Dataset

def to_ds(rows):
    return Dataset.from_dict({
        "src": [str(r.get(DIFF_KEY, "")) for r in rows],
        "tgt": [str(r.get(GOLD_KEY, "")) for r in rows],
    })

ds = {k: to_ds(v) for k, v in splits.items() if v}
print({k: len(v) for k, v in ds.items()})


## 2. Tokenize & load model


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)   # works on transformers 4.44.2
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

def preprocess(batch):
    enc = tokenizer(batch["src"], max_length=MAX_SRC, truncation=True)
    lab = tokenizer(text_target=batch["tgt"], max_length=MAX_TGT, truncation=True)
    enc["labels"] = lab["input_ids"]
    return enc

tok = {k: v.map(preprocess, batched=True, remove_columns=["src", "tgt"])
       for k, v in ds.items()}
print("tokenized:", {k: len(v) for k, v in tok.items()})


## 3. Fine-tune


In [ ]:
from transformers import (DataCollatorForSeq2Seq, Seq2SeqTrainer,
                              Seq2SeqTrainingArguments)

collator = DataCollatorForSeq2Seq(tokenizer, model=model)
fp16 = torch.cuda.is_available()

args = Seq2SeqTrainingArguments(
    output_dir=str(CKPT_DIR),
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=EVAL_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=50,
    predict_with_generate=True,
    generation_max_length=MAX_TGT,
    fp16=fp16,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=tok["train"],
    eval_dataset=tok.get("valid") or tok.get("test"),
    data_collator=collator, tokenizer=tokenizer,
)
train_result = trainer.train()
print(train_result.metrics)


## 4. Evaluate — BLEU-4 on the test split

**Metric note:** the CodeReviewer/CodeXGLUE `msg` task reports **smoothed** BLEU-4.
Plain `sacrebleu` corpus-BLEU near-zeros out on short, single-reference comments
(no 4-gram match → the geometric mean collapses), so it is NOT comparable to the
paper's number. We report smoothed BLEU-4 as the primary metric and keep sacrebleu
only as a secondary reference.


In [ ]:
import numpy as np
import sacrebleu
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction

test_ds = ds.get("test") or ds.get("valid")
preds = trainer.predict(tok["test"] if "test" in tok else tok["valid"],
                        num_beams=GEN_BEAMS, max_length=MAX_TGT)
pred_ids = preds.predictions
pred_ids = np.where(pred_ids != -100, pred_ids, tokenizer.pad_token_id)
hyps = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
refs = [ex for ex in test_ds["tgt"]]

# Primary: smoothed BLEU-4 (paper-comparable). Secondary: sacrebleu (reference only).
sm = SmoothingFunction().method4
refs_tok = [r.split() for r in refs]
hyps_tok = [h.split() for h in hyps]
BLEU4 = 100 * sum(sentence_bleu([r], h, smoothing_function=sm)
                  for r, h in zip(refs_tok, hyps_tok)) / len(hyps_tok)
bleu_corpus_sm = corpus_bleu([[r] for r in refs_tok], hyps_tok, smoothing_function=sm) * 100
bleu_sacre = sacrebleu.corpus_bleu(hyps, [refs]).score

print(f"BLEU-4 (smoothed, sentence-avg) : {BLEU4:.2f}   <- primary, comparable to paper 14.8")
print(f"BLEU-4 (smoothed, corpus)       : {bleu_corpus_sm:.2f}")
print(f"BLEU-4 (sacrebleu, reference)   : {bleu_sacre:.2f}")


### CodeBLEU (with caveat)

CodeBLEU expects **code** as hypothesis/reference (AST + data-flow match). Review
comments are natural language, so this number is not very meaningful for the `msg`
task — reported only for completeness against the stated ≥22 target. Treat BLEU-4
above as the real result. If CodeBLEU is genuinely required, it belongs on the
**code-refinement (`ref`)** task, which we'd add as a separate track.


In [ ]:
try:
    from codebleu import calc_codebleu
    # Language arg required; comments aren't code, so this is nominal. Use 'python'.
    cb = calc_codebleu(refs, hyps, lang="python")
    print("CodeBLEU (nominal):", {k: round(v, 4) for k, v in cb.items()})
except Exception as e:
    print("CodeBLEU skipped:", repr(e))


## 5. Results table + export weights & example outputs


In [ ]:
PAPER_BLEU4 = 14.8
TARGET_BLEU4, TARGET_CODEBLEU = 18.0, 22.0
rows = [
    ("BLEU-4", f"{BLEU4:.2f}", f"paper {PAPER_BLEU4} / target ≥{TARGET_BLEU4}",
     "PASS" if BLEU4 >= TARGET_BLEU4 else "below target"),
]
print(f"{'metric':10s} {'actual':8s} {'reference':28s} {'verdict'}")
for m, a, r, v in rows:
    print(f"{m:10s} {a:8s} {r:28s} {v}")

# Export fine-tuned weights.
trainer.save_model(str(WEIGHTS_DIR)); tokenizer.save_pretrained(str(WEIGHTS_DIR))
print("\nSaved weights ->", WEIGHTS_DIR)


In [ ]:
# "After" examples (pair with Phase 1 baseline for the before/after report table).
n = min(5, len(test_ds))
after = [{"diff": test_ds["src"][i], "gold_comment": test_ds["tgt"][i],
          "finetuned_pred": hyps[i]} for i in range(n)]
out = ART_DIR / "phase2_finetuned_examples.json"
with open(out, "w", encoding="utf-8") as fh:
    json.dump({"model": MODEL_NAME, "bleu4_smoothed": BLEU4,
               "train_size": len(ds["train"]), "epochs": EPOCHS,
               "examples": after}, fh, indent=2)
print("Saved ->", out)
for i, ex in enumerate(after, 1):
    print("=" * 90)
    print(f"[{i}] GOLD:", ex["gold_comment"][:200])
    print("    PRED:", ex["finetuned_pred"][:200])


## 6. (Optional) Publish weights to Hugging Face Hub — for the live demo

The demo notebook (`03_demo_inference.ipynb`) loads the model by id from the Hub,
so the student can run it on their own Colab account with **no dataset download and
no Drive access**. Do this once after a good fine-tune.

**One-time setup (human, not automated):** create a free account at huggingface.co,
then a **write** token under *Settings → Access Tokens*. Set `PUSH_TO_HF = True` and
`HF_MODEL_ID`, run the cell, and paste the token when prompted (it is not stored).


In [ ]:
PUSH_TO_HF  = False                                  # flip to True when ready
HF_MODEL_ID = "your-username/codet5p-220m-msg-pyjs"  # <-- set your HF id

if PUSH_TO_HF:
    from huggingface_hub import login
    from getpass import getpass
    login(getpass("HF write token: "))               # entered by you, not stored
    model.push_to_hub(HF_MODEL_ID)
    tokenizer.push_to_hub(HF_MODEL_ID)
    print("Pushed ->", f"https://huggingface.co/{HF_MODEL_ID}")
else:
    print("PUSH_TO_HF is False. Weights are local at:", WEIGHTS_DIR)
    print("Set PUSH_TO_HF=True (+ an HF write token) to host them for 03_demo_inference.ipynb.")


## Next: report the numbers back

Send the printed **BLEU-4** (and CodeBLEU if computed), the **train/valid/test
sizes**, the **training config** (epochs / LR / batch / time), and the before/after
examples. These go into the Phase 1 & 2 BUILD_LOG entries and the results table.

If BLEU-4 < 18: options are (a) train on the full set (`MAX_TRAIN=None`),
(b) more epochs, (c) scale to `codet5p-770m`. That's the expected post-Aug-2
optimization pass.
